In [1]:
import os
from utils import extract_body, tokenize, clean_tokens, decode
from utils import chunk_tokens, flatten_token_chunks
from utils import extract_few_shot_examples
from utils import select_few_shot 
from utils import merge_tokens_with_auto_labels, add_style_and_parent_to_auto_labels, compare_html_allow_auto_labels
from models import GPTAssistant
from process_chunks import process_chunks

In [2]:
# ---------- Define Hyperparameters ----------
min_tokens = 500
model_name = "gpt-4.1"

n_few_shot = 10  # Number of few-shot examples to use

#### Define the text to process, and where to save it. Define the text for few shot

In [5]:
# File paths
project_root = r"C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process"
filename = "1997CanLII16226_ONCA"
anno = "llm"
version = "v1"
html_path = fr"{project_root}\data\Document_Échantillon_Initial\ronde_2\plain_html_arbre_balise\{filename}.htm"
output_dir = fr"{project_root}\data\Documents_Annotés\{anno}\{filename}"

os.makedirs(output_dir, exist_ok=True)

# Read HTML file
with open(html_path, 'r', encoding='utf-8') as file:
    html_content = file.read()
print(f"   ✓ HTML file loaded: {html_path}")


fs_filename = "1999CanLII7320_annotated"
fs_anno = "EG"
fs_version = "v1"
fs_html_path = fr"{project_root}\data\Documents_Annotés\{fs_anno}\{fs_filename}_{fs_anno}_{fs_version}.html"
# Read HTML file
with open(fs_html_path, 'r', encoding='utf-8') as file:
    fs_html_content = file.read()
print(f"   ✓ HTML file loaded for few shot: {fs_html_path}")


   ✓ HTML file loaded: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Document_Échantillon_Initial\ronde_2\plain_html_arbre_balise\1997CanLII16226_ONCA.htm
   ✓ HTML file loaded for few shot: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\EG\1999CanLII7320_annotated_EG_v1.html


### Process The HTML Content

In [6]:

# ---------- Extract body content ----------
body_content = extract_body(html_content)


# ---------- Tokenize body content ----------
tokens = tokenize(body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunks = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)


   ⚠ Warning: stop_bookmark_separation=True but bookmark not found
   ✓ Chunked tokens into 179 chunks (>= 500 tokens each)


In [7]:
# ---------- Extract body content ----------
fs_body_content = extract_body(fs_html_content)


# ---------- Tokenize body content ----------
fs_tokens = tokenize(fs_body_content)

# ---------- Clean tokens ----------
normalized_cleaned_tokens = clean_tokens(html_tokens=fs_tokens, normalize=True, keep_manual_label=True, keep_bookmarks=True)

# ---------- Chunk tokens ----------
token_chunk1, token_chunk2 = chunk_tokens(normalized_cleaned_tokens, min_tokens=min_tokens, stop_bookmark_separation=True)

   ✓ Found bookmark separator at index 23066
   ✓ Splitting: 23066 tokens before, 31001 tokens after
   ✓ Chunked tokens into 36 chunks (>= 500 tokens each)
   ✓ Chunked tokens into 62 chunks (>= 500 tokens each)
   ✓ Total chunks: 36 before + 62 after = 98


In [8]:
label_config = {
    "keep_attributes":["labelname"], # extraction only, no disambiguation
    "switch_type":True, # manual_label -> auto_label
    "use_simplified":True, # <auto_label labelname="title"> -> <title>
    "keep_labels":["decision", "legislation", "secondary sources"]
}

In [9]:
# ---------- Create few-shot examples ----------

few_shot_examples = extract_few_shot_examples(token_chunk1, 
                                              label_config)



selected_few_shot_examples = select_few_shot(examples=few_shot_examples, n=n_few_shot)
print(f"   ✓ Selected {len(selected_few_shot_examples)} few-shot examples for processing.")

   ✓ Extracted 36 few-shot examples from chunks
   ✓ Selected 10 few-shot examples for processing.


In [10]:
# ---------- Initialize LLM model ----------
model = GPTAssistant(model_name)

In [11]:
# ---------- Process chunks ----------

prompt_path = fr"{project_root}\llm_based_annotation\utils\prompts\simplified_parent_extraction_cot.txt"

processed_chunks = process_chunks(
    model=model,
    token_chunks=token_chunks,
    process_prompt_path=prompt_path,
    label_config=label_config,
    few_shot_examples=selected_few_shot_examples,
    output_dir=output_dir,
    filename=filename
)


   ✓ Processing 179 chunks with LLM...
   ✓ Using 10 few-shot examples


Processing chunks:   0%|          | 0/179 [00:00<?, ?it/s]

Processing chunks:   1%|          | 1/179 [00:08<25:38,  8.64s/it]

   → Step 1: Tokenized into 518 tokens
   ✓ Extracted 516 tokens between <start> and <end>
   → Step 2: Extracted 516 tokens between markers
   ✓ Converted 516 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks:   1%|          | 2/179 [00:14<21:28,  7.28s/it]

...

   → Step 1: Tokenized into 509 tokens
   ✓ Extracted 507 tokens between <start> and <end>
   → Step 2: Extracted 507 tokens between markers
   ✓ Converted 507 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks:  99%|█████████▉| 178/179 [19:02<00:04,  4.26s/it]

   → Step 1: Tokenized into 506 tokens
   ✓ Extracted 504 tokens between <start> and <end>
   → Step 2: Extracted 504 tokens between markers
   ✓ Converted 504 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format


Processing chunks: 100%|██████████| 179/179 [19:06<00:00,  6.40s/it]

   → Step 1: Tokenized into 173 tokens
   ✓ Extracted 171 tokens between <start> and <end>
   → Step 2: Extracted 171 tokens between markers
   ✓ Converted 171 tokens from simplified to auto_label format
   → Step 3: Converted to auto_label format
   ✓ Processing history saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\history_1997CanLII16226_ONCA.json

   ✓ Processing completed:
      - Total chunks: 179
      - Successful: 179
      - Failed: 0
   ✓ Processed chunks saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA\processed_chunks_1997CanLII16226_ONCA.json


## Post Processing

In [15]:
# ---------- Merge all tokens ----------
processed_tokens_flat = flatten_token_chunks(processed_chunks)


original_tokens = tokenize(html_content)
processed_html_content = merge_tokens_with_auto_labels(original_tokens, processed_tokens_flat)

processed_html = decode(processed_html_content)

print(f"\nMerged HTML length: {len(processed_html)}")

# ---------- Add style and parent to auto_label tags ----------
processed_html_content = decode(add_style_and_parent_to_auto_labels(processed_html))


# ---------- Compare with original HTML (ignoring auto_label tags) ----------
comparison_result = compare_html_allow_auto_labels(processed_html_content, html_content)



   ✓ Flattened 179 chunks into 90476 tokens

Merged HTML length: 283753
   ✓ HTMLs match when ignoring auto_label tags


In [16]:
# ---------- Save processed HTML to file ----------
with open(fr"{output_dir}\{filename}_llm_{anno}.html", 'w', encoding='utf-8') as f:
    f.write(processed_html_content)
print(f"   ✓ Processed HTML saved to: {output_dir}")

   ✓ Processed HTML saved to: C:\Users\zakga\OneDrive\Documents\code\LeREaD_annotation_process\data\Documents_Annotés\llm\1997CanLII16226_ONCA
